# G17 – Entrenamiento de modelos con MLflow en Databricks

Este notebook usa `src/modelado.py` del repositorio, de modo que todos los integrantes entrenan con el mismo
preprocesamiento, la misma validación temporal y las mismas métricas.

**Antes de correrlo**
1. `modelado.py` debe estar en el mismo Volume que la base (o en `../src` si se usa el repositorio como Git folder).
2. `dataset_modelado.csv` debe estar cargado en un Volume (por defecto `/Volumes/workspace/default/g17/`).
3. Arriba, en los widgets, elija su modelo y el desenlace.

In [0]:
%pip install -q catboost xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.dropdown("modelo", "elegir", ["elegir", "logistica", "random_forest", "xgboost", "catboost"])
dbutils.widgets.dropdown("desenlace", "estancia_prolongada", ["estancia_prolongada", "mortality"])
dbutils.widgets.dropdown("variables", "tablero", ["tablero", "completo"])
dbutils.widgets.text("ruta_datos", "/Volumes/workspace/default/g17/dataset_modelado.csv")
dbutils.widgets.text("registrar_como", "")  # p. ej. workspace.default.g17_estancia (solo el modelo elegido)

In [0]:
import os
import sys

import mlflow

# Busca modelado.py en ../src (repositorio como Git folder) o en el Volume donde está la base
carpeta_nb = os.getcwd()
raiz_repo = os.path.abspath(os.path.join(carpeta_nb, ".."))
carpeta_volume = os.path.dirname(dbutils.widgets.get("ruta_datos"))
en_repo = os.path.exists(os.path.join(raiz_repo, "src", "modelado.py"))
sys.path.insert(0, os.path.join(raiz_repo, "src") if en_repo else carpeta_volume)
import modelado  # noqa: E402
print("modelado.py cargado desde:", modelado.__file__)

usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/G17_UCI_triage"
mlflow.set_registry_uri("databricks-uc")
print("Usuario:", usuario)
print("Experimento:", experimento)

modelado.py cargado desde: /Volumes/workspace/default/g17/modelado.py
Usuario: a.ardilas@uniandes.edu.co
Experimento: /Users/a.ardilas@uniandes.edu.co/G17_UCI_triage


In [0]:
if dbutils.widgets.get("modelo") == "elegir":
    dbutils.notebook.exit("Elija su modelo en el menú 'modelo' de arriba y vuelva a darle Run all.")

resultado = modelado.entrenar(
    modelo=dbutils.widgets.get("modelo"),
    desenlace=dbutils.widgets.get("desenlace"),
    variables=dbutils.widgets.get("variables"),
    datos=dbutils.widgets.get("ruta_datos"),
    experimento=experimento,
    autor=usuario,
    registrar_como=dbutils.widgets.get("registrar_como") or None,
)
resultado

  {'iterations': 500, 'depth': 4, 'learning_rate': 0.03} -> AUROC CV 0.944 ± 0.014
  {'iterations': 500, 'depth': 4, 'learning_rate': 0.1} -> AUROC CV 0.941 ± 0.016
  {'iterations': 500, 'depth': 6, 'learning_rate': 0.03} -> AUROC CV 0.946 ± 0.016
  {'iterations': 500, 'depth': 6, 'learning_rate': 0.1} -> AUROC CV 0.942 ± 0.017
  {'iterations': 1000, 'depth': 4, 'learning_rate': 0.03} -> AUROC CV 0.939 ± 0.016
  {'iterations': 1000, 'depth': 4, 'learning_rate': 0.1} -> AUROC CV 0.940 ± 0.016
  {'iterations': 1000, 'depth': 6, 'learning_rate': 0.03} -> AUROC CV 0.944 ± 0.016
  {'iterations': 1000, 'depth': 6, 'learning_rate': 0.1} -> AUROC CV 0.942 ± 0.017


🔗 View Logged Model at: https://dbc-c661b0e5-cfb5.cloud.databricks.com/ml/experiments/3290427598514491/models/m-35646e994d2b4a13b57981ce4f033005?o=7474644747663168



catboost | mortality | completo: mejor {'iterations': 500, 'depth': 6, 'learning_rate': 0.03}
  AUROC CV 0.946 | AUROC prueba 0.918 (0.852-0.968) | Brier 0.064 | E/O 0.94


{'run_id': '17d62cf7e90b46d2b94eca996dc4f296',
 'mejores_hiperparametros': {'iterations': 500,
  'depth': 6,
  'learning_rate': 0.03},
 'test_auroc': 0.9179936305732485,
 'test_auroc_ic_inf': 0.8524803443666954,
 'test_auroc_ic_sup': 0.9684988125685057,
 'test_auprc': 0.6815385274891643,
 'test_brier': 0.06410446480715186,
 'test_calib_intercepto': -0.24007899009697162,
 'test_calib_pendiente': 0.6030761704658731,
 'test_prob_media': 0.12481750777901748,
 'test_tasa_observada': 0.13259668508287292,
 'test_razon_esperado_observado': 0.9413320378334236,
 'test_pct_eventos_en_top20': 79.16666666666667,
 'test_pct_dias_cama_en_top20': 39.88919667590028,
 'test_auroc_saps3_mismos_pacientes': 0.8810897435897436,
 'test_auroc_modelo_mismos_pacientes': 0.9423076923076923,
 'cv_auroc_mejor': 0.9462896708595135}

## Comparar las corridas
Tabla de las corridas principales del experimento (una por modelo × desenlace × conjunto de variables).

In [0]:
corridas = mlflow.search_runs(experiment_names=[experimento])

if "tags.mlflow.parentRunId" in corridas.columns:
    corridas = corridas[corridas["tags.mlflow.parentRunId"].isna()]

columnas = [
    "tags.modelo",
    "tags.desenlace",
    "tags.variables",
    "metrics.cv_auroc_mejor",
    "metrics.test_auroc",
    "metrics.test_auroc_ic_inf",
    "metrics.test_auroc_ic_sup",
    "metrics.test_brier",
    "metrics.test_razon_esperado_observado",
    "metrics.test_pct_eventos_en_top20",
    "metrics.test_pct_dias_cama_en_top20",
    "metrics.test_auroc_saps3_mismos_pacientes"
]

tabla = corridas[
    [c for c in columnas if c in corridas.columns]
].copy()

# Eliminar ejecuciones sin resultados
tabla = tabla.dropna(subset=["metrics.test_auroc"])

# Conservar una ejecución por combinación
tabla = tabla.drop_duplicates(
    subset=["tags.modelo", "tags.desenlace", "tags.variables"],
    keep="first"
)

# Ordenar por desempeño
tabla = tabla.sort_values(
    "metrics.test_auroc",
    ascending=False
)

display(tabla)

tags.modelo,tags.desenlace,tags.variables,metrics.cv_auroc_mejor,metrics.test_auroc,metrics.test_auroc_ic_inf,metrics.test_auroc_ic_sup,metrics.test_brier,metrics.test_razon_esperado_observado,metrics.test_pct_eventos_en_top20,metrics.test_pct_dias_cama_en_top20,metrics.test_auroc_saps3_mismos_pacientes
xgboost,mortality,completo,0.939422972956738,0.944267515923567,0.90173500064553,0.9757380351130351,0.06327408709807811,1.1200897401819627,83.33333333333333,39.473684210526315,0.8810897435897436
xgboost,mortality,tablero,0.9370517518251098,0.9397558386411891,0.8991374734607218,0.9734032808525709,0.06661186524332906,1.142172104989489,83.33333333333333,40.443213296398895,0.8810897435897436
catboost,mortality,tablero,0.9398054150598092,0.9211783439490446,0.8697911076267946,0.9636133262449051,0.07223347618551293,1.046149051198436,75.0,38.08864265927978,0.8810897435897436
catboost,mortality,completo,0.9462896708595135,0.9179936305732485,0.8524803443666954,0.9684988125685057,0.06410446480715186,0.9413320378334236,79.16666666666667,39.88919667590028,0.8810897435897436
xgboost,estancia_prolongada,completo,0.8669810238765511,0.9062049062049061,0.8546086713120284,0.9483827129577946,0.08438078768063804,0.9919744470605143,74.07407407407408,48.199445983379505,0.7413621675366642
xgboost,estancia_prolongada,tablero,0.8679635074549056,0.898027898027898,0.8511313769781511,0.9414118318560302,0.08959729631667177,1.1057306627432504,66.66666666666667,48.337950138504155,0.7413621675366642
catboost,estancia_prolongada,tablero,0.8774256073402148,0.8953823953823954,0.8473398631417498,0.9381898454746137,0.09331920628864054,1.0959465948382585,66.66666666666667,46.26038781163435,0.7413621675366642
catboost,estancia_prolongada,completo,0.8795413408403713,0.8944203944203943,0.8377615412835467,0.9412803956529122,0.09007847845170151,0.7867131202427481,66.66666666666667,46.81440443213297,0.7413621675366642


## Guardar el resumen en el repositorio
Escribe `resultados_<usuario>.csv` en el Volume (o en `reports/modelos/` si se usa el Git folder).
Descárguelo y súbalo al repositorio en `reports/modelos/` (ver la guía).

In [0]:
carpeta = os.path.join(raiz_repo, "reports", "modelos") if en_repo else carpeta_volume
os.makedirs(carpeta, exist_ok=True)
nombre = usuario.split("@")[0].replace(".", "_")
salida = os.path.join(carpeta, f"resultados_{nombre}.csv")
tabla.rename(columns=lambda c: c.split(".", 1)[-1]).to_csv(salida, index=False)
print("Guardado:", salida)